# Module 1: Cross-Sectional Analysis & Relative Value

- **Industry Landscape**: High-level overview of broad market sectors.
- **Stock vs. Industry Relative Value**: Zooming into a single constituent to reveal stationary mean-reverting spreads.
- **Industry Universe Screening**: Ranking constituents cross-sectionally.
- **Dollar-Neutral Strategy**: Long Top 10 / Short Bottom 10 equal-capital backtest.
- **Beta-Neutral Strategy**: Eliminating systematic benchmark exposure via dynamic beta weighting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import yfinance as yf

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

### 1. Broad Industry Overview

- Tracks performance across major US industry sectors using liquid SPDR ETFs.
- Uses log prices to linearize compounding growth and ensure scale independence.
- **Normalized Log Price Formula**: 
  $$\ln\left(\frac{P_t}{P_0}\right) = \ln(P_t) - \ln(P_0)$$

In [ ]:
sector_tickers = ['XLK', 'XLF', 'XLV', 'XLE', 'XLI', 'XLY', 'XLP', 'XLU', 'XLB', 'XLC', 'XLRE']
sector_names = {
    'XLK': 'Technology',
    'XLF': 'Financials',
    'XLV': 'Health Care',
    'XLE': 'Energy',
    'XLI': 'Industrials',
    'XLY': 'Consumer Discretionary',
    'XLP': 'Consumer Staples',
    'XLU': 'Utilities',
    'XLB': 'Materials',
    'XLC': 'Communication',
    'XLRE': 'Real Estate'
}

sector_raw = yf.download(sector_tickers, start='2021-01-01', end='2025-01-01', auto_adjust=True, progress=False)['Close'].ffill().dropna()
log_sectors = np.log(sector_raw) - np.log(sector_raw.iloc[0])

fig, ax = plt.subplots(figsize=(14, 7))
cmap = plt.cm.get_cmap('tab20', len(sector_tickers))
for i, t in enumerate(sector_tickers):
    ax.plot(log_sectors.index, log_sectors[t], label=f"{t} ({sector_names[t]})", color=cmap(i), lw=1.8)

ax.set_title('Major US Industry Sectors: Cumulative Log Price Performance', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Normalized Log Price: ln(P_t / P_0)', fontsize=11)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

### 2. Zooming In: Stock vs. Industry Relative Performance

- Examines an individual stock (MSFT) relative to its parent industry ETF (XLK) in log prices.
- Strips out common sector beta via OLS regression in log space:
  $$\ln(P_{\text{MSFT}, t}) = \alpha + \beta \ln(P_{\text{XLK}, t}) + \epsilon_t$$
  $$\text{Spread}_t = \ln(P_{\text{MSFT}, t}) - (\alpha + \beta \ln(P_{\text{XLK}, t}))$$
- Standardizes the spread into z-score:
  $$Z_t = \frac{\text{Spread}_t - \mu}{\sigma}$$
- Evaluates mean reversion half-life via Ornstein-Uhlenbeck / AR(1) process:
  $$\Delta \text{Spread}_t = \mu + \lambda \text{Spread}_{t-1}, \quad H = -\frac{\ln(2)}{\lambda}$$

In [ ]:
stock_pair_raw = yf.download(['MSFT', 'XLK'], start='2021-01-01', end='2025-01-01', auto_adjust=True, progress=False)['Close'].ffill().dropna()

log_stock = np.log(stock_pair_raw['MSFT'])
log_industry = np.log(stock_pair_raw['XLK'])
x_industry = sm.add_constant(log_industry)

#IMPORTANT
ols_fit = sm.OLS(log_stock, x_industry).fit()
alpha = ols_fit.params['const']
beta = ols_fit.params['XLK']

#IMPORTANT
residual_spread = log_stock - (alpha + beta * log_industry)

#IMPORTANT
z_spread = (residual_spread - residual_spread.mean()) / residual_spread.std()

#IMPORTANT
adf_stat, p_val, _, _, crit_vals, _ = adfuller(residual_spread)

delta_spread = np.diff(residual_spread)
lagged_spread = sm.add_constant(residual_spread.iloc[:-1].values)

#IMPORTANT
ou_fit = sm.OLS(delta_spread, lagged_spread).fit()
lambda_val = ou_fit.params[1]
half_life = -np.log(2) / lambda_val if lambda_val < 0 else np.nan

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios': [1, 1.2]})

ax1.plot(stock_pair_raw.index, log_stock, label='MSFT ln(Price)', color='#1f77b4', lw=2)
ax1.plot(stock_pair_raw.index, alpha + beta * log_industry, label=f'Fitted Model ({alpha:.2f} + {beta:.2f} * ln(XLK))', color='#ff7f0e', ls='--', lw=2)
ax1.set_title('Log Price vs. Industry-Regressed Level: ln(MSFT) vs. ln(XLK)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Log Price', fontsize=11)
ax1.legend(loc='upper left')

ax2.plot(residual_spread.index, residual_spread, color='#2ca02c', lw=1.8, label=f'Log Residual Spread: ln(MSFT) - ({beta:.2f} * ln(XLK))')
ax2.axhline(residual_spread.mean(), color='black', ls='--', lw=1.2, label='Mean Spread')
ax2.axhline(residual_spread.mean() + residual_spread.std(), color='red', ls=':', label='±1σ Threshold')
ax2.axhline(residual_spread.mean() - residual_spread.std(), color='red', ls=':')
ax2.axhline(residual_spread.mean() + 2 * residual_spread.std(), color='darkred', ls=':', lw=1.5, label='±2σ Threshold')
ax2.axhline(residual_spread.mean() - 2 * residual_spread.std(), color='darkred', ls=':', lw=1.5)

diag_box = f"Hedge Ratio (β): {beta:.3f}\nADF Statistic: {adf_stat:.3f}\nADF p-value: {p_val:.4e}\nHalf-Life: {half_life:.1f} days"
ax2.text(0.02, 0.93, diag_box, transform=ax2.transAxes, fontsize=10, va='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9))

ax2.set_title('Log Relative Performance Spread (Stationary Mean-Reverting Residual)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('Log Spread', fontsize=11)
ax2.legend(loc='lower left')

plt.tight_layout()
plt.show()

### 3. Industry Universe Construction & Diagnostic Table

- Gathers a 30-stock liquid universe across the Technology industry.
- Calculates beta, correlation, and spread stationarity in log space.
- **Formulas**:
  - Industry Beta: $$\beta_i = \frac{\text{Cov}(R_i, R_{\text{XLK}})}{\text{Var}(R_{\text{XLK}})}$$
  - Correlation: $$\text{Corr}(R_i, R_{\text{XLK}}) = \frac{\text{Cov}(R_i, R_{\text{XLK}})}{\sigma_i \sigma_{\text{XLK}}}$$
  - Annualized Volatility: $$\sigma_{\text{ann}} = \sigma_{\text{daily}} \times \sqrt{252}$$

In [ ]:
industry_tickers = [
    'AAPL', 'MSFT', 'NVDA', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'CSCO', 'AMD', 'INTC',
    'QCOM', 'TXN', 'IBM', 'AMAT', 'NOW', 'LRCX', 'ADI', 'MU', 'KLAC', 'SNPS',
    'CDNS', 'MCHP', 'PANW', 'FTNT', 'ANET', 'MSI', 'NXPI', 'TEL', 'APH', 'GLW'
]

all_fetch_tickers = industry_tickers + ['XLK', 'SPY']
full_data = yf.download(all_fetch_tickers, start='2021-01-01', end='2025-01-01', auto_adjust=True, progress=False)['Close'].ffill().dropna()

stock_prices = full_data[industry_tickers]
xlk_prices = full_data['XLK']
spy_prices = full_data['SPY']

daily_stock_ret = stock_prices.pct_change().dropna()
daily_xlk_ret = xlk_prices.pct_change().dropna()

summary_metrics = []
for t in industry_tickers:
    y = np.log(stock_prices[t])
    X = sm.add_constant(np.log(xlk_prices))
    #IMPORTANT
    res = sm.OLS(y, X).fit()
    b = res.params['XLK']
    spr = y - (res.params['const'] + b * np.log(xlk_prices))
    p_stat, p_val, _, _, _, _ = adfuller(spr)
    corr = daily_stock_ret[t].corr(daily_xlk_ret)
    vol = daily_stock_ret[t].std() * np.sqrt(252) * 100
    tot_ret = (stock_prices[t].iloc[-1] / stock_prices[t].iloc[0] - 1) * 100
    summary_metrics.append({
        'Ticker': t,
        'Industry Beta': round(b, 2),
        'Correlation to XLK': round(corr, 3),
        'Spread ADF Stat': round(p_stat, 2),
        'Spread ADF p-val': f"{p_val:.3e}",
        'Annualized Vol (%)': round(vol, 1),
        'Total Return (%)': round(tot_ret, 1)
    })

industry_summary_df = pd.DataFrame(summary_metrics).sort_values('Correlation to XLK', ascending=False).reset_index(drop=True)
industry_summary_df

### 4. Cross-Sectional Ranking: Top 10 vs. Bottom 10

- Freezes time at a single rebalance date to rank all 30 constituents.
- **Ranking Formula**:
  $$\text{Rank}_{i, t} = \text{Rank}(R_{i, t}), \quad i \in \{1, \dots, N\}$$
- Highlights the Top 10 relative leaders (long candidates) and Bottom 10 relative laggards (short candidates).

In [ ]:
monthly_stock_prices = stock_prices.resample('ME').last()
monthly_stock_ret = monthly_stock_prices.pct_change().dropna()
sample_month = monthly_stock_ret.index[-1]

#IMPORTANT
month_returns = monthly_stock_ret.loc[sample_month].sort_values(ascending=False) * 100

bar_colors = ['#2ca02c' if i < 10 else ('#d62728' if i >= 20 else '#7f7f7f') for i in range(len(month_returns))]

fig, ax = plt.subplots(figsize=(14, 8))
bars = ax.barh(month_returns.index, month_returns.values, color=bar_colors, edgecolor='black', linewidth=0.7)
ax.axvline(0, color='black', lw=1.2)
ax.axvline(month_returns.mean(), color='blue', ls='--', lw=1.8, label=f'Industry Average ({month_returns.mean():.2f}%)')

for bar in bars:
    w = bar.get_width()
    x_pos = w + 0.3 if w >= 0 else w - 1.2
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f"{w:+.1f}%", va='center', fontsize=9, fontweight='bold')

ax.set_title(f"Cross-Sectional Rank Slice ({sample_month.strftime('%B %Y')}): Top 10 Long vs. Bottom 10 Short", fontsize=13, fontweight='bold')
ax.set_xlabel('1-Month Return (%)', fontsize=11)
ax.set_ylabel('Stock Ticker', fontsize=11)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 5. Dollar-Neutral Strategy Backtest

- Equal dollar weighting: +50% capital long Top 10, -50% capital short Bottom 10.
- **Formulas**:
  - Equal-Weighted Leg Returns: $$R_{\text{long}, t} = \frac{1}{10}\sum_{i \in \text{Top 10}} R_{i, t}, \quad R_{\text{short}, t} = \frac{1}{10}\sum_{j \in \text{Bottom 10}} R_{j, t}$$
  - Dollar-Neutral Return: $$R_{\text{Dollar-Neutral}, t} = 0.5 \times R_{\text{long}, t} - 0.5 \times R_{\text{short}, t}$$
  - Cumulative Growth of $1.00: $$V_t = \prod_{\tau=1}^{t} (1 + R_{\tau})$$
  - Peak-to-Trough Drawdown: $$\text{DD}_t = \frac{V_t}{\max_{0 \le \tau \le t} V_{\tau}} - 1$$

In [ ]:
monthly_signal = monthly_stock_ret.shift(1).dropna()
aligned_ret = monthly_stock_ret.loc[monthly_signal.index]

monthly_xlk_prices = xlk_prices.resample('ME').last()
monthly_spy_prices = spy_prices.resample('ME').last()

aligned_xlk = monthly_xlk_prices.pct_change().dropna().loc[monthly_signal.index]
aligned_spy = monthly_spy_prices.pct_change().dropna().loc[monthly_signal.index]

top_baskets = monthly_signal.apply(lambda row: row.nlargest(10).index.tolist(), axis=1)
bottom_baskets = monthly_signal.apply(lambda row: row.nsmallest(10).index.tolist(), axis=1)

#IMPORTANT
long_leg_ret = pd.Series([aligned_ret.loc[dt, top_baskets[dt]].mean() for dt in aligned_ret.index], index=aligned_ret.index)
short_leg_ret = pd.Series([aligned_ret.loc[dt, bottom_baskets[dt]].mean() for dt in aligned_ret.index], index=aligned_ret.index)

#IMPORTANT
ls_strategy_ret = 0.5 * long_leg_ret - 0.5 * short_leg_ret

#IMPORTANT
cum_ls = (1 + ls_strategy_ret).cumprod()
cum_long = (1 + long_leg_ret).cumprod()
cum_short = (1 + short_leg_ret).cumprod()
cum_xlk = (1 + aligned_xlk).cumprod()
cum_spy = (1 + aligned_spy).cumprod()

#IMPORTANT
dd_ls = (cum_ls / cum_ls.cummax() - 1) * 100
dd_xlk = (cum_xlk / cum_xlk.cummax() - 1) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(cum_ls.index, cum_ls, label='Dollar-Neutral Spread (50% Long Top 10 / 50% Short Bottom 10)', color='#2ca02c', lw=2.5)
ax1.plot(cum_xlk.index, cum_xlk, label='XLK Industry ETF (Long Only)', color='#ff7f0e', lw=1.8, ls='--')
ax1.plot(cum_spy.index, cum_spy, label='SPY Market Benchmark', color='black', lw=1.5, ls=':')
ax1.plot(cum_long.index, cum_long, label='Long Top 10 Only', color='#1f77b4', lw=1.2, alpha=0.6)
ax1.plot(cum_short.index, cum_short, label='Short Bottom 10 Only', color='#d62728', lw=1.2, alpha=0.6)
ax1.set_title('Cross-Sectional Dollar-Neutral Backtest: Growth of $1.00', fontsize=13, fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)', fontsize=11)
ax1.legend(loc='upper left', frameon=True)

ax2.fill_between(dd_ls.index, dd_ls, 0, color='#2ca02c', alpha=0.4, label='Dollar-Neutral Drawdown (%)')
ax2.plot(dd_xlk.index, dd_xlk, color='#ff7f0e', lw=1.5, ls='--', label='XLK Benchmark Drawdown (%)')
ax2.set_title('Drawdown Comparison: Dollar-Neutral Hedged Spread vs. Unhedged Industry ETF', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('Drawdown (%)', fontsize=11)
ax2.legend(loc='lower left', frameon=True)

plt.tight_layout()
plt.show()

### 6. Beta-Neutral Strategy: Eliminating Systematic Market Risk

- **Dollar Neutral vs. Beta Neutral**: A dollar neutral does not guard against positive and negitive beta. Its not technicaly neutral to the overall market. : $\beta_{\text{net}} = \beta_{\text{long}} - \beta_{\text{short}} \ne 0$.
- **Beta-Neutral Condition**: Set net portfolio sensitivity to the industry/market benchmark to zero:
  $$\beta_{\text{portfolio}} = w_{\text{long}} \beta_{\text{long}} - w_{\text{short}} \beta_{\text{short}} = 0$$
- **Dynamic Capital Weights**: Weight each leg inversely to its beta:
  $$w_{\text{long}, t} = \frac{\beta_{\text{short}, t}}{\beta_{\text{long}, t} + \beta_{\text{short}, t}}, \quad w_{\text{short}, t} = \frac{\beta_{\text{long}, t}}{\beta_{\text{long}, t} + \beta_{\text{short}, t}}$$
- **Beta-Neutral Strategy Return**: 
  $$R_{\text{Beta-Neutral}, t} = w_{\text{long}, t} R_{\text{long}, t} - w_{\text{short}, t} R_{\text{short}, t}$$

In [ ]:
rolling_cov = daily_stock_ret.rolling(63).cov(daily_xlk_ret)
rolling_var = daily_xlk_ret.rolling(63).var()

#IMPORTANT
rolling_betas = rolling_cov.div(rolling_var, axis=0)
monthly_betas = rolling_betas.resample('ME').last().shift(1).dropna()

common_dates = aligned_ret.index.intersection(monthly_betas.index)

beta_long_series = []
beta_short_series = []
for dt in common_dates:
    b_long = monthly_betas.loc[dt, top_baskets[dt]].mean()
    b_short = monthly_betas.loc[dt, bottom_baskets[dt]].mean()
    beta_long_series.append(b_long)
    beta_short_series.append(b_short)

beta_long_series = pd.Series(beta_long_series, index=common_dates)
beta_short_series = pd.Series(beta_short_series, index=common_dates)

#IMPORTANT
w_long = beta_short_series / (beta_long_series + beta_short_series)
w_short = beta_long_series / (beta_long_series + beta_short_series)

#IMPORTANT
beta_neutral_ret = w_long * long_leg_ret.loc[common_dates] - w_short * short_leg_ret.loc[common_dates]
dollar_neutral_ret_common = ls_strategy_ret.loc[common_dates]

#IMPORTANT
cum_beta_neutral = (1 + beta_neutral_ret).cumprod()
cum_dollar_neutral = (1 + dollar_neutral_ret_common).cumprod()
cum_bench = (1 + aligned_xlk.loc[common_dates]).cumprod()

#IMPORTANT
dd_beta_neutral = (cum_beta_neutral / cum_beta_neutral.cummax() - 1) * 100
dd_dollar_neutral = (cum_dollar_neutral / cum_dollar_neutral.cummax() - 1) * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(cum_beta_neutral.index, cum_beta_neutral, label='Beta-Neutral Strategy (Zero Industry Beta)', color='#1f77b4', lw=2.5)
ax1.plot(cum_dollar_neutral.index, cum_dollar_neutral, label='Dollar-Neutral Strategy (50/50 Capital)', color='#2ca02c', lw=2, ls='--')
ax1.plot(cum_bench.index, cum_bench, label='XLK Benchmark (Long Only)', color='#ff7f0e', lw=1.5, ls=':')
ax1.set_title('Strategy Comparison: Beta-Neutral vs. Dollar-Neutral Cumulative Performance', fontsize=13, fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)', fontsize=11)
ax1.legend(loc='upper left', frameon=True)

ax2.plot(dd_beta_neutral.index, dd_beta_neutral, label='Beta-Neutral Drawdown (%)', color='#1f77b4', lw=1.8)
ax2.plot(dd_dollar_neutral.index, dd_dollar_neutral, label='Dollar-Neutral Drawdown (%)', color='#2ca02c', lw=1.8, ls='--')
ax2.set_title('Drawdown Comparison: Impact of Beta Elimination', fontsize=12, fontweight='bold')
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('Drawdown (%)', fontsize=11)
ax2.legend(loc='lower left', frameon=True)

plt.tight_layout()
plt.show()

neutral_comparison_table = pd.DataFrame({
    'Strategy / Benchmark': ['Beta-Neutral Spread', 'Dollar-Neutral Spread', 'XLK (Technology ETF)'],
    'Annualized Return (%)': [
        round(beta_neutral_ret.mean() * 12 * 100, 2),
        round(dollar_neutral_ret_common.mean() * 12 * 100, 2),
        round(aligned_xlk.loc[common_dates].mean() * 12 * 100, 2)
    ],
    'Annualized Volatility (%)': [
        round(beta_neutral_ret.std() * np.sqrt(12) * 100, 2),
        round(dollar_neutral_ret_common.std() * np.sqrt(12) * 100, 2),
        round(aligned_xlk.loc[common_dates].std() * np.sqrt(12) * 100, 2)
    ],
    'Sharpe Ratio (Rf=0%)': [
        round((beta_neutral_ret.mean() * 12) / (beta_neutral_ret.std() * np.sqrt(12)), 2),
        round((dollar_neutral_ret_common.mean() * 12) / (dollar_neutral_ret_common.std() * np.sqrt(12)), 2),
        round((aligned_xlk.loc[common_dates].mean() * 12) / (aligned_xlk.loc[common_dates].std() * np.sqrt(12)), 2)
    ],
    'Max Drawdown (%)': [
        round(dd_beta_neutral.min(), 2),
        round(dd_dollar_neutral.min(), 2),
        round(((cum_bench / cum_bench.cummax() - 1) * 100).min(), 2)
    ]
})

neutral_comparison_table

### Module 1 Key Takeaways

- **Industry Normalization**: Regressing stocks against their industry ETF in log prices strips out broad factor drift and yields stationary spreads.
- **Cross-Sectional Ranking**: Slicing constituents point-in-time isolates relative winners and losers without directional market bias.
- **Dollar Neutrality vs. Beta Neutrality**: Dollar neutrality hedges raw invested capital; beta neutrality dynamically scales position sizing to achieve true zero-sensitivity to market and sector shocks.